# LAB 09 — Bias, Variance and Imbalanced Data 
## Overfitting, ROC/AUC, Resampling, SMOTE and Noisy Data on Real Medical Data 

## Student Information
- **Name:Abdullah Hussain**
- **Student ID:023-24-0241**
- **Section:G**
- **GitHub:https:https://github.com/Abdullah-Hussain-45**
- **Kaggle:https:https://www.kaggle.com/abdullahhussain34**
- **Dataset:** Health_Care Dataset Stroke (Kaggle)

---


## Part 0 — Setup and the Data  (10 minutes) 
ML Lab 09 — Bias, Variance and Imbalanced Data 
A hospital wants to flag patients at risk of stroke from routine information. The target column is stroke (1 = had a 
stroke, 0 = did not). 
1. Install imbalanced-learn (pip install imbalanced-learn) and import everything you need. 
2. Load the CSV. Report the shape, the columns with missing values, the value counts of gender and the 
number of patients with stroke = 1 and stroke = 0 (count and percentage). 
3. Clean the data: remove the identifier column; remove any gender category that has too few rows to learn or 
test; apply categorical encoding to the text columns (pd.get_dummies, as in the labs). 
4. Split 80/20 with stratify=y and random_state=SEED. Fill the missing numeric values with a median learned 
from the TRAIN set only. Create a scaled copy (StandardScaler fitted on train) to use for Logistic Regression. 
5. Report how many stroke cases are in the training set and in the test set. 

In [13]:
# 2.
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load the dataset
df = pd.read_csv("A:\\Semester 5\\ML\\healthcare-dataset-stroke-data_1.csv")
SEED = 42

# Shape of the dataset
print(df.shape)

# Missing Values from cols
missing_cols = df.columns[df.isnull().sum() > 0]
print(df[missing_cols].isnull().sum())

# Gender Counts
gender_counts = df['gender'].value_counts(dropna=False)
print(gender_counts)

# No. of Patients with Stroke(0,1)
percentage =  df['stroke'].value_counts(normalize=True) * 100 
stroke_Counts = df['stroke'].value_counts()
print(f"Counts of Stroke: {stroke_Counts}",end=" ")
print(f"Percentage % : {percentage} ")

# 3. Clean the data
# dropping the column
df_clean = df.drop(columns=['id'])

# dropping the gender column with other that has one row
df_clean = df_clean[df_clean['gender'] != 'Other'].copy()

# One-Hot Encoding
categorical_cols = df_clean.select_dtypes(include=['object','string']).columns.tolist()
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True, dtype=int)

# Question-4: Stratified Split & Imputation
X = df_encoded.drop(columns=['stroke'])
y = df_encoded['stroke'] 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)

# Filling missing values with Train median set only
nums_cols = X_train.select_dtypes(include=[np.number]).columns
train_Medians = X_test[nums_cols].median()

X_train[nums_cols] = X_train[nums_cols].fillna(train_Medians)
X_test[nums_cols] = X_test[nums_cols].fillna(train_Medians)

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[nums_cols] = scaler.fit_transform(X_train[nums_cols])
X_test_scaled[nums_cols] = scaler.transform(X_test[nums_cols])

# Question 5: Report Stroke Cases
print("\nStroke Cases in Splits:")
print(f"Train set: {sum(y_train == 1)} stroke cases out of {len(y_train)} ({sum(y_train == 1)/len(y_train):.2%})")
print(f"Test set:  {sum(y_test == 1)} stroke cases out of {len(y_test)} ({sum(y_test == 1)/len(y_test):.2%})")




(5110, 12)
bmi    201
dtype: int64
gender
Female    2994
Male      2115
Other        1
Name: count, dtype: int64
Counts of Stroke: stroke
0    4861
1     249
Name: count, dtype: int64 Percentage % : stroke
0    95.127202
1     4.872798
Name: proportion, dtype: float64 

Stroke Cases in Splits:
Train set: 199 stroke cases out of 4087 (4.87%)
Test set:  50 stroke cases out of 1022 (4.89%)


Question 
(a) In 2-3 lines: how imbalanced is the data? Why did we use stratify=y, and what could go wrong with only a 
few stroke cases in the test set? 


Answer:
The dataset is heavily imbalanced, with only 249 stroke cases out of 5,110 patients (~4.87% positive class vs. 95.13% negative class). We used stratify=y to force the 80/20 train/test split to preserve this exact 4.87% proportion in both subsets. Without stratification, a random split could assign almost no stroke cases to the test set, making performance evaluation (like recall, precision, and ROC-AUC) completely unreliable due to extreme sample variance.

## Part 1 — The Accuracy Trap  (15 minutes) 
Objective 
Show that a model can have excellent accuracy and still be useless. 
1. Build a baseline that always predicts the majority class (DummyClassifier, strategy='most_frequent'). Report 
accuracy. 
2. Train a default Logistic Regression on the scaled training data. Report Accuracy, Precision, Recall, F1, ROC 
AUC and PR AUC (average precision) on the test set, and plot or print its confusion matrix. 
3. Write a small helper function that takes a model and train/test data and returns these six metrics - you will 
reuse it all lab. 
(a) Compare the accuracy of the baseline and of Logistic Regression. What do you conclude? 
(b) Look at the confusion matrix and the recall. In the hospital's situation, what does a false negative mean, and 
which numbers should we trust instead of accuracy? 

## Part 2 — Underfitting, Overfitting, Bias and Variance  (40 minutes) 
A Decision Tree is a good model to study this because one parameter, max_depth, moves it smoothly from too 
simple to too complex. From now on use ROC AUC (not accuracy) as the score for cross-validation. 
Task 2.1 — Validation curve 
1. For max_depth = 1, 2, ..., 15 and None, record the train ROC AUC and the 5-fold stratified cross-validation 
ROC AUC (train data only). 
2. Plot both curves on one chart (labelled axes, title, legend). Report the best depth by CV AUC. 
Task 2.2 — Learning curves 
1. Use sklearn.model_selection.learning_curve (scoring='roc_auc', train_sizes from 20% to 100%) for a tree with 
max_depth=1 and a tree with max_depth=None. Plot train and CV AUC against the number of training rows, 
side by side. 
Task 2.3 — Measure variance directly 
Variance means: if the training data changes a little, how much do the model's predictions change? Measure it. 
1. For each of max_depth = 1, 3 and None: draw 40 bootstrap samples of the TRAIN rows (sample with 
replacement, same size), fit a tree on each, and collect each tree's predicted stroke probability for every test 
patient. 
2. For every test patient, compute the variance of its 40 predicted probabilities; then average over patients. 
Also report the mean test ROC AUC of the 40 trees and the ROC AUC of the averaged prediction (mean of the 
40 probability vectors). 
3. Show the three depths in one table. 
Hints 
** •  rng.choice(len(X_train), len(X_train), replace=True) gives bootstrap row positions; use 
X_train.iloc[idx]. 
probs.var(axis=0). 
•  Collect predictions in an array of shape (40, number of test patients); variance across models is 
•  learning_curve returns (train_sizes, train_scores, valid_scores) - average the scores over the 
folds with .mean(axis=1). 
(a) From the validation curve: which depths underfit, which overfit, and which depth would you choose? Quote 
train and CV AUC for one depth in each zone. 
(b) From the learning curves: how can you tell high bias from high variance? Will adding more training rows fix 
either one? 
(c) From the variance table: how does variance change with depth? What happens to the AUC of the averaged 
prediction compared with the single trees? Link this to what Random Forest does (Lab 4). 
(d) In your own words: what is the bias-variance trade-off, and where do depth 1, depth 3 and depth None sit?